# Data and evaluation


In [ ]:
from pathlib import Path

ELEMENT_TYPES = [1, 2]  # [1] triangular, [2] quadrilateral, or [1, 2] both
OUTPUT_PATH = Path("data/smoothing_data/vasonet_mixed_dataset.npz")
N_PATCHES = 60_000  # Per type
REGENERATE = False
SEED = 42
SWEEPS = 10

In [ ]:
import numpy as np
import torch
from scipy.optimize import minimize
from scipy.special import logsumexp


class Patch:
    """CCW boundary of a fan with v incident cells.

    Triangles have v boundary vertices. Quadrilaterals have 2*v because
    each face adds its corner opposite the center.
    """

    def __init__(self, ring, kind):
        self.ring = np.asarray(ring, dtype=np.float64)
        self.kind = kind
        self.valence = len(ring) // kind
        self.lower = self.ring.min(axis=0)
        self.scale = np.ptp(self.ring, axis=0).max()

    @staticmethod
    def faces(valence, kind):
        first = 1 + kind * np.arange(valence)
        columns = [np.zeros_like(first), first]
        if kind == 2:
            columns.append(first + 1)
        return np.column_stack((*columns, np.roll(first, -1)))

    def encode(self, shifts=(0,), *, width=None):
        width = 18 * self.kind if width is None else width
        indices = (np.arange(len(self.ring)) + self.kind * np.asarray(shifts)[:, None]) % len(self.ring)
        coordinates = np.zeros((len(indices), width // 2, 2))
        coordinates[:, :len(self.ring)] = (self.ring[indices] - self.lower) / self.scale
        return coordinates.reshape(len(indices), -1)

    def predict(self, model_patch):
        parameter = next(model_patch.parameters())
        options = dict(dtype=parameter.dtype, device=parameter.device)
        coordinates = torch.as_tensor(self.encode(), **options)
        valence = torch.tensor([[self.valence]], **options)
        with torch.inference_mode():
            prediction = model_patch(coordinates, valence).cpu().numpy()
        return self.lower + self.scale * np.asarray(prediction, dtype=np.float64).reshape(2)


class Extract:
    """Recover each interior vertex's CCW boundary from oriented mesh faces."""

    def __init__(self, faces):
        self.links = {}
        for face in faces:
            for i, node in enumerate(face):
                ordered = np.roll(face, -i)
                self.links.setdefault(node, {})[ordered[1]] = ordered[1:]
        edges = np.stack((faces, np.roll(faces, -1, axis=1)), axis=-1).reshape(-1, 2)
        edges, counts = np.unique(np.sort(edges, axis=1), axis=0, return_counts=True)
        self.free_nodes = np.setdiff1d(np.unique(faces), np.unique(edges[counts == 1]))

    def ring(self, node):
        links = self.links[node]
        start = current = min(links)
        ring = []
        for _ in links:
            face = links[current]
            ring.extend(face[:-1])
            current = face[-1]
            if current == start:
                break
        if current != start or len(ring) != len(links) * (len(face) - 1):
            raise ValueError(f"Vertex {node} has no single closed one-ring")
        return np.asarray(ring)

In [ ]:
class Optimization:
    def __init__(self, kind):
        self.kind = kind
        self.quality = self.triangle if kind == 1 else self.quadrilateral

    @staticmethod
    def triangle(center, ring):
        """Regularized mean ratio and its analytic derivatives"""
        u = ring - center
        v = np.roll(u, -1, axis=0)

        # S=[u, (2v-u)/sqrt(3)].
        norm2 = np.sum(u*u, axis=1) + np.sum((2*v-u)**2, axis=1) / 3.0
        det = (2.0 / np.sqrt(3.0)) * (u[:, 0]*v[:, 1] - u[:, 1]*v[:, 0])
        root = np.sqrt(det**2 + 4*2.5e-5**2)
        numerator = np.where(det >= 0, det+root, 4*2.5e-5**2/(root-det))
        q = numerator / norm2

        d_norm2 = -(4.0/3.0) * (u+v)
        a = ring
        b = np.roll(ring, -1, axis=0)
        d_det = (2.0 / np.sqrt(3.0)) * np.column_stack((a[:, 1]-b[:, 1], b[:, 0]-a[:, 0]))
        d_numerator = (1 + det/root)[:, None] * d_det
        d_q = d_numerator/norm2[:, None] - (q/norm2)[:, None] * d_norm2
        return q, det, d_q


    @staticmethod
    def corners(quads):
        outgoing = np.roll(quads, -1, axis=-2) - quads
        incoming = np.roll(quads, 1, axis=-2) - quads
        jacobian = np.stack((outgoing, incoming), axis=-1)
        det = np.linalg.det(jacobian)
        length2 = np.sum(jacobian**2, axis=(-2, -1))
        q = np.divide(2 * det, length2, out=np.zeros_like(det), where=length2 > 0)
        return q, det, length2, outgoing, incoming


    @staticmethod
    def quadrilateral(center, ring):
        q, det, length2, outgoing, incoming = Optimization.corners(np.vstack((center, ring))[Patch.faces(len(ring) // 2, 2)])

        # The derivative of each directed edge is ±I if it touches the center.
        d_outgoing = np.array([-1, 0, 0, 1])[:, None]
        d_incoming = np.array([-1, 1, 0, 0])[:, None]
        d_det = (
            d_outgoing * np.stack((incoming[..., 1], -incoming[..., 0]), axis=-1)
            + d_incoming * np.stack((-outgoing[..., 1], outgoing[..., 0]), axis=-1)
        )
        d_length2 = 2 * (d_outgoing * outgoing + d_incoming * incoming)
        d_q = (2 * d_det - q[..., None] * d_length2) / length2[..., None]
        return q, det, d_q

    def objective(self, center, ring):
        q, _, gradient = self.quality(center, ring)
        z = -50 * q.ravel()
        norm = logsumexp(z)
        return norm / 50, -np.exp(z - norm) @ gradient.reshape(-1, 2)

    def solve(self, center, ring):
        result = minimize(self.objective, np.zeros(2), args=(ring - center,),
                          method="CG", jac=True, options={"gtol": 1e-7, "maxiter": 300})
        return center + result.x

    def valid(self, det, ring):
        if self.kind == 1:
            return np.all(det > 2e-12 / np.sqrt(3))
        return np.all(det > 1e-12 * np.ptp(ring, axis=0).max()**2)

    def mesh_quality(self, vertices, faces):
        elements = vertices[faces]
        if self.kind == 2:
            return self.corners(elements)[0].min(axis=1)
        a, b, c = elements.transpose(1, 0, 2)
        u, v = b - a, c - a
        det = u[:, 0] * v[:, 1] - u[:, 1] * v[:, 0]
        length2 = np.sum(u**2 + v**2 + (u - v)**2, axis=1)
        return 2 * np.sqrt(3) * det / length2

## Dataset generation

A mixed NPZ stores both types together. The existing VASONet training notebook accepts single-type NPZs.

In [ ]:
from tempfile import NamedTemporaryFile


def generate_patches(kind, count, width, group_offset, seed):
    rng = np.random.default_rng(seed)
    opt = Optimization(kind)
    valences = np.arange(3, 10)
    probabilities = np.exp(-0.5 * ((valences - 5.43) / 1.62)**2)
    probabilities /= probabilities.sum()
    # Synthetic fans use the same CCW extraction as the evaluation meshes.
    order = {v: Extract(Patch.faces(v, kind)).ring(0) - 1 for v in valences}
    origin = np.zeros(2)
    rotation = (-np.pi / 2, np.pi / 2) if kind == 1 else (0, 2 * np.pi)

    n_train, n_val = 7 * count // 10, 2 * count // 10
    split = np.empty(count, dtype=np.int8)
    split[np.random.default_rng(seed).permutation(count)] = np.repeat(
        np.arange(3), [n_train, n_val, count - n_train - n_val]
    )

    coordinates, counts, centers, qualities = [], [], [], []
    for _ in range(count):
        v = int(rng.choice(valences, p=probabilities))
        n = kind * v
        step = 2 * np.pi / n
        for _ in range(500 if kind == 1 else 2000):
            theta = step * np.arange(n) + rng.uniform(-0.5 * step, 0.5 * step, n)
            theta += rng.uniform(*rotation)
            radius = 1 - rng.uniform(-0.9, 0.9, n)
            ring = (radius[:, None] * np.column_stack((np.cos(theta), np.sin(theta))))[order[v]]
            before, det, _ = opt.quality(origin, ring)
            if not opt.valid(det, ring):
                continue
            target = opt.solve(origin, ring)
            after, det, _ = opt.quality(target, ring)
            if kind == 2 and (not opt.valid(det, ring)
                              or opt.objective(target, ring)[0] > opt.objective(origin, ring)[0] + 1e-12):
                continue
            break
        else:
            raise RuntimeError(f"No acceptable valence-{v} patch")

        patch = Patch(ring, kind)
        coordinates.append(patch.encode(np.arange(v), width=width))
        counts.append(v)
        centers.append((np.stack((origin, target)) - patch.lower) / patch.scale)
        qualities.append((before.min(), after.min()))

    initial, target = np.repeat(centers, counts, axis=0).transpose(1, 0, 2)
    before, after = np.repeat(qualities, counts, axis=0).T
    return dict(
        coordinates=np.concatenate(coordinates),
        valence=np.repeat(counts, counts).astype(float)[:, None],
        group_id=np.repeat(np.arange(group_offset, group_offset + count, dtype=np.int32), counts),
        split=np.repeat(split, counts), center_initial=initial, target=target,
        min_quality_before=before, min_quality_after=after,
    )


def generate_dataset(element_types, count, path, seed):
    width = 18 * max(element_types)
    blocks = [generate_patches(kind, count, width, i * count, seed)
              for i, kind in enumerate(element_types)]
    data = {key: np.concatenate([block[key] for block in blocks]) for key in blocks[0]}
    if len(element_types) > 1:
        data["element_type"] = np.concatenate([
            np.full(len(block["valence"]), kind, dtype=np.int8)
            for kind, block in zip(element_types, blocks)
        ])

    path.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(dir=path.parent, suffix=".npz", delete=False) as handle:
        temporary = Path(handle.name)
    try:
        np.savez_compressed(temporary, **data)
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)
    print(f"{path} | {count * len(element_types):,} patches | {len(data['target']):,} rows")

In [ ]:
element_types = tuple(dict.fromkeys(ELEMENT_TYPES))
if not element_types or any(kind not in (1, 2) for kind in element_types):
    raise ValueError("ELEMENT_TYPES must select triangles (1), quadrilaterals (2), or both.")

if REGENERATE or not OUTPUT_PATH.is_file():
    generate_dataset(element_types, N_PATCHES, OUTPUT_PATH, SEED)
else:
    with np.load(OUTPUT_PATH) as data:
        saved_types = (set(np.unique(data["element_type"])) if "element_type" in data
                       else {data["coordinates"].shape[1] // 18})
    if saved_types != set(element_types):
        raise ValueError(f"{OUTPUT_PATH} contains element types {sorted(saved_types)}. "
                         "Choose a different OUTPUT_PATH or set REGENERATE = True.")
    print(f"Using {OUTPUT_PATH}")

## Evaluation

Choose the checkpoints below. To evaluate existing weights, you can skip the dataset writing cell.

In [ ]:
# Set a path to None to run only the baselines for that element type.
TRI_MODEL_PATH = Path("MLP_tri.pt")
QUAD_MODEL_PATH = Path("MLP_quad.pt")

In [ ]:
from time import perf_counter

from vasonet import VASONet

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection


def load_evaluation_model(path, kind):
    if path is None:
        return None
    weights = torch.load(path, map_location="cpu", weights_only=True)
    width = 18 * kind
    if weights["geo_mlp.0.weight"].shape[1] != width or weights["topo_mlp.0.weight"].shape[1] != 1:
        raise ValueError(f"{path} is incompatible with the VASONet for element type {kind}.")
    model = VASONet(width).to(dtype=weights["geo_mlp.0.weight"].dtype)
    model.load_state_dict(weights)
    return model.eval()


def evaluate(meshes, kind, model_patch):
    opt = Optimization(kind)
    methods = {"Laplacian": lambda center, ring: ring.mean(axis=0), "PR-CG": opt.solve}
    if model_patch is not None:
        def predict(center, ring):
            patch = Patch(ring, kind)
            return patch.predict(model_patch) if 3 <= patch.valence <= 9 else opt.solve(center, ring)

        methods["NN"] = predict

    rows = ["| Case | Method | Minimum quality | Mean quality | Invalid cells | Time (s) |",
            "|---|---|---:|---:|---:|---:|"]
    fig, axes = plt.subplots(len(meshes), len(methods) + 1, squeeze=False,
                             figsize=(3 * (len(methods) + 1), 3 * len(meshes)),
                             layout="constrained")
    cmap = plt.get_cmap("RdYlBu").copy()
    cmap.set_under("#B218B2")
    results = {}
    for row, (name, mesh) in enumerate(meshes.items()):
        original, faces = mesh["V"], mesh["F"]
        topology = Extract(faces)
        rings = {i: topology.ring(i) for i in topology.free_nodes}
        results[name] = {"Original": original}
        times = {"Original": "—"}
        for method, move in methods.items():
            vertices = original.copy()
            start = perf_counter()
            for _ in range(SWEEPS):
                for i, ring in rings.items():
                    vertices[i] = move(vertices[i], vertices[ring])
            results[name][method] = vertices
            times[method] = f"{perf_counter() - start:.3f}"
        for col, (method, vertices) in enumerate(results[name].items()):
            q = opt.mesh_quality(vertices, faces)
            rows.append(f"| {name} | {method} | {q.min():.4f} | {q.mean():.4f} | {(q <= 0).sum()} | {times[method]} |")
            ax = axes[row, col]
            image = PolyCollection(vertices[faces], array=q, cmap=cmap, clim=(0, 1),
                                   edgecolors="0.25", linewidths=0.22)
            ax.add_collection(image)
            ax.autoscale_view()
            ax.set(aspect="equal", xticks=[], yticks=[])
            if row == 0:
                ax.set_title(method)
            if col == 0:
                ax.set_ylabel(name)
        print(f"{name}: completed {SWEEPS} sweeps", flush=True)
    fig.colorbar(image, ax=axes, extend="min", label="Mean-ratio quality", shrink=0.75)
    display(Markdown("\n".join(rows)))
    plt.show()
    return results

### Triangular cases

In [ ]:
from matplotlib.path import Path as PolygonPath
from scipy.spatial import Delaunay


def sample_polygon(corners, spacing):
    """Sample straight edges without duplicating their shared endpoints."""
    corners = np.asarray(corners, dtype=float)
    edges = []
    for start, end in zip(corners, np.roll(corners, -1, axis=0)):
        n = int(np.ceil(np.linalg.norm(end - start) / spacing))
        edges.append(start + np.arange(n)[:, None] / n * (end - start))
    return np.vstack(edges)


def ellipse_boundary(center, axes, angle, spacing):
    """Polygonal approximation; angle is in radians."""
    n = int(np.ceil(2 * np.pi * max(axes) / spacing))
    theta = np.linspace(0, 2 * np.pi, n, endpoint=False)
    points = np.column_stack((np.cos(theta), np.sin(theta))) * axes
    c, s = np.cos(angle), np.sin(angle)
    return points @ np.array([[c, s], [-s, c]]) + center


def build_test_mesh(outer, holes, n_interior, seed):
    """Recover polygon boundary edges before classifying Delaunay faces."""
    loops = [outer, *holes]
    paths = [PolygonPath(loop) for loop in loops]

    def inside(points):
        mask = paths[0].contains_points(points)
        for path in paths[1:]:
            mask &= ~path.contains_points(points)
        return mask

    mesh_rng = np.random.default_rng(seed)
    lower, upper = outer.min(axis=0), outer.max(axis=0)
    interior = np.empty((0, 2))
    while len(interior) < n_interior:
        candidates = mesh_rng.uniform(lower, upper, size=(2 * n_interior, 2))
        interior = np.vstack((interior, candidates[inside(candidates)]))
    interior = interior[:n_interior]

    for _ in range(12):
        n_boundary = sum(map(len, loops))
        V = np.vstack([*loops, interior])
        F = Delaunay(V).simplices
        edges = np.sort(F[:, [[0, 1], [1, 2], [2, 0]]].reshape(-1, 2), axis=1)
        edge_set = set(map(tuple, edges))
        missing = []
        boundary_edges = []
        offset = 0
        for loop in loops:
            ids = np.arange(offset, offset + len(loop))
            segments = [tuple(sorted((i, j))) for i, j in zip(ids, np.roll(ids, -1))]
            boundary_edges.extend(segments)
            missing.append(np.array([edge not in edge_set for edge in segments]))
            offset += len(loop)
        if not any(mask.any() for mask in missing):
            break
        # Midpoints lie on the same polygon, so refinement preserves geometry.
        refined = []
        for loop, mask in zip(loops, missing):
            points = []
            for start, end, split in zip(loop, np.roll(loop, -1, axis=0), mask):
                points.append(start)
                if split:
                    points.append((start + end) / 2)
            refined.append(np.asarray(points))
        loops = refined
    else:
        raise RuntimeError("Boundary recovery failed; use a constrained mesher.")

    # Once all boundary segments are mesh edges, no face can cross a hole.
    F = F[inside(V[F].mean(axis=1))].copy()
    a, b, c = V[F[:, 0]], V[F[:, 1]], V[F[:, 2]]
    signed_area2 = (b[:, 0] - a[:, 0]) * (c[:, 1] - a[:, 1]) - (b[:, 1] - a[:, 1]) * (c[:, 0] - a[:, 0])
    flipped = signed_area2 < 0
    F[flipped] = F[flipped][:, [0, 2, 1]]

    edges, counts = np.unique(
        np.sort(F[:, [[0, 1], [1, 2], [2, 0]]].reshape(-1, 2), axis=1),
        axis=0, return_counts=True,
    )
    boundary_mask = np.zeros(len(V), dtype=bool)
    boundary_mask[np.unique(edges[counts == 1])] = True
    assert set(map(tuple, edges[counts == 1])) == set(boundary_edges), "Boundary segments changed."
    assert np.all(np.abs(signed_area2) > 1e-14), "Degenerate triangle."
    assert counts.max() == 2, "Non-manifold mesh."
    assert np.array_equal(np.flatnonzero(boundary_mask), np.arange(n_boundary)), "Unexpected boundary."
    assert len(np.unique(F)) == len(V), "Unused mesh vertices."
    return dict(V=V, F=F, boundary_mask=boundary_mask,
                free_nodes=np.flatnonzero(~boundary_mask))

In [ ]:
model_patch = load_evaluation_model(TRI_MODEL_PATH, kind=1)

h = 0.10

square = sample_polygon([[0, 0], [1, 0], [1, 1], [0, 1]], h)

theta = np.pi / 2 + np.arange(10) * np.pi / 5
radii = np.where(np.arange(10) % 2 == 0, 0.45, 0.22)
star = sample_polygon(radii[:, None] * np.column_stack((np.cos(theta), np.sin(theta))), h)
circle = ellipse_boundary((0, 0), (1, 1), 0, h)

plate = sample_polygon([[0, 0], [2, 0], [2, 2], [0, 2]], h)
holes = [
    ellipse_boundary((0.70, 0.70), (0.48, 0.48), 0, h),
    ellipse_boundary((1.40, 1.40), (0.62, 0.25), -np.pi / 4, h),
]

meshes = {
    "Unstructured Square": build_test_mesh(square, [], 180, SEED),
    "Circle with Star Hole": build_test_mesh(circle, [star], 450, SEED + 1),
    "Perforated Plate": build_test_mesh(plate, holes, 400, SEED + 2),
}

tri_results = evaluate(meshes, kind=1, model_patch=model_patch)

### Quadrilateral case

In [ ]:
model_patch = load_evaluation_model(QUAD_MODEL_PATH, kind=2)

n = 11
axis = np.linspace(0, 1, n)
vertices = np.column_stack([a.ravel() for a in np.meshgrid(axis, axis)])
grid = np.arange(n * n).reshape(n, n)
faces = np.column_stack([a.ravel() for a in
                         (grid[:-1, :-1], grid[:-1, 1:], grid[1:, 1:], grid[1:, :-1])])
interior = grid[1:-1, 1:-1].ravel()
rng = np.random.default_rng(SEED)
vertices[interior] += rng.uniform(-0.18, 0.18, (len(interior), 2)) / (n - 1)
quad_meshes = {"Perturbed square": {"V": vertices, "F": faces}}
quad_results = evaluate(quad_meshes, kind=2, model_patch=model_patch)